# SOEA-Plus (PDEMC): Post-Decisional Metacognitive Control Benchmark

**Author:** Haifaa Owayed  
**Track:** Metacognition  
**Competition:** Measuring Progress Toward AGI - Cognitive Abilities

## Overview

SOEA-Plus (PDEMC) is a three-task benchmark that measures **Post-Decisional Metacognitive Control** in LLMs.

Unlike calibration benchmarks that measure confidence at decision time, SOEA-Plus evaluates what happens **after** the decision:

- **Task 1 — Decision:** Can the model correctly classify a medical claim? (Accuracy + SOCE)
- **Task 2 — Monitoring:** After deciding, can the model estimate its own error probability? (Monitoring Accuracy)
- **Task 3 — Control:** Given its uncertainty, does the model choose the rational action? (Control Rationality)

### The Control Collapse Hypothesis
> *Models do not fail at knowing — they fail at acting on uncertainty.*

**Dataset:** 300 real PubMed claim-evidence pairs, gold-annotated by Haifaa Owayed.


## Setup

In [ ]:
# Install required packages
!pip install kaggle-benchmarks -q
!pip install pandas -q

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import json
from dataclasses import dataclass
from typing import Optional

print("kaggle-benchmarks loaded successfully")
print(f"Default LLM: {kbench.llm}")

## Load Dataset

300 real PubMed claim-evidence pairs, gold-annotated.

In [ ]:
# Load the SOEA-Plus dataset
# The dataset is embedded directly for reproducibility
import io

# Load from Kaggle dataset input (attach SOEA_300_gold_FINAL.csv as dataset)
try:
    df = pd.read_csv('/kaggle/input/soea-plus-dataset/SOEA_300_gold_FINAL.csv')
    print(f"Loaded from Kaggle dataset: {len(df)} examples")
except FileNotFoundError:
    # Fallback: load from notebook directory
    try:
        df = pd.read_csv('SOEA_300_gold_FINAL.csv')
        print(f"Loaded from local file: {len(df)} examples")
    except FileNotFoundError:
        print("ERROR: Please attach the SOEA_300_gold_FINAL.csv dataset to this notebook")
        print("Dataset available at: https://github.com/Haifawaeedd/SOEA-Benchmark")
        raise

print(f"Label distribution:")
print(df['gold_label'].value_counts())
df.head(2)

## Task 1: Decision

**Measures:** Can the model correctly classify a medical claim as SUPPORTED, REFUTED, or INCONCLUSIVE based on evidence?

**Metrics:**
- Accuracy (correct label)
- SOCE (Sign of Confidence Error) — does confidence increase when wrong?
- ECE (Expected Calibration Error)

In [ ]:
@dataclass
class DecisionOutput:
    label: str          # SUPPORTED, REFUTED, or INCONCLUSIVE
    confidence: float   # 0.0 to 1.0
    reasoning: str      # Brief justification


@kbench.task(name="soea_task1_decision")
def task1_decision(llm, claim: str, evidence: str, gold_label: str) -> float:
    """
    Task 1: Post-Decisional Metacognition — Decision Phase
    
    Given a medical claim and supporting evidence from PubMed,
    classify the claim as SUPPORTED, REFUTED, or INCONCLUSIVE.
    Return a confidence score (0.0-1.0).
    
    Returns accuracy score (1.0 if correct, 0.0 if wrong).
    """
    prompt = f"""You are a medical evidence evaluator. Analyze the following claim and evidence.

CLAIM: {claim}

EVIDENCE: {evidence}

Classify the relationship between the claim and evidence as exactly one of:
- SUPPORTED: The evidence clearly supports the claim
- REFUTED: The evidence clearly contradicts the claim  
- INCONCLUSIVE: The evidence is ambiguous, insufficient, or only partially relevant

Respond with a JSON object containing:
- label: one of SUPPORTED, REFUTED, INCONCLUSIVE
- confidence: your confidence from 0.0 to 1.0
- reasoning: one sentence explaining your decision"""

    result = llm.prompt(prompt, schema=DecisionOutput)
    
    # Normalize label
    predicted = result.label.upper().strip()
    if predicted not in ['SUPPORTED', 'REFUTED', 'INCONCLUSIVE']:
        predicted = 'INCONCLUSIVE'
    
    is_correct = (predicted == gold_label.upper().strip())
    
    # Assert with informative message
    kbench.assertions.assert_true(
        is_correct,
        expectation=f"Model predicted '{predicted}' but gold label is '{gold_label}'. Confidence: {result.confidence:.2f}"
    )
    
    return 1.0 if is_correct else 0.0


print("Task 1 defined: soea_task1_decision")

## Task 2: Post-Decision Monitoring

**Measures:** After making a decision, can the model accurately estimate the probability that its answer was wrong?

**Key insight:** This is evaluated *after* Task 1. The model has already committed to an answer. Now we ask: how likely is that answer to be incorrect?

**Metric:** Monitoring Accuracy — does the model assign higher error probability when it was actually wrong?

In [ ]:
@dataclass
class MonitoringOutput:
    error_probability: float  # 0.0 to 1.0 — how likely is the answer wrong?
    uncertainty_reason: str   # What evidence makes you uncertain?
    key_ambiguity: str        # The main source of ambiguity


@kbench.task(name="soea_task2_monitoring")
def task2_monitoring(llm, claim: str, evidence: str, model_answer: str) -> float:
    """
    Task 2: Post-Decisional Metacognition — Monitoring Phase
    
    After the model has classified a claim, it must estimate the probability
    that its own answer was incorrect. This tests metacognitive awareness.
    
    CRITICAL: No new information is provided. The model must reason about
    its own uncertainty based solely on the original claim and evidence.
    
    Returns monitoring score: 1.0 if error_probability > 0.5 (model is uncertain),
    0.0 if error_probability <= 0.5 (model is overconfident).
    """
    prompt = f"""You previously analyzed a medical claim and gave an answer. Now reflect on that decision.

ORIGINAL CLAIM: {claim}

EVIDENCE YOU REVIEWED: {evidence}

YOUR PREVIOUS ANSWER: {model_answer}

Now perform post-decisional monitoring:
1. How likely is it that your answer was INCORRECT? (0.0 = very confident correct, 1.0 = very likely wrong)
2. What specific aspects of the evidence made you uncertain?
3. What is the main source of ambiguity in this case?

Be honest about your uncertainty. Medical evidence is often ambiguous.

Respond with a JSON object containing:
- error_probability: float from 0.0 to 1.0
- uncertainty_reason: one sentence about what made you uncertain
- key_ambiguity: the main source of ambiguity"""

    result = llm.prompt(prompt, schema=MonitoringOutput)
    
    # Clamp to valid range
    ep = max(0.0, min(1.0, result.error_probability))
    
    # A model showing meaningful uncertainty (ep > 0.3) demonstrates monitoring awareness
    shows_uncertainty = ep > 0.3
    
    kbench.assertions.assert_true(
        shows_uncertainty,
        expectation=f"Model should show meaningful uncertainty (ep > 0.3). Got ep={ep:.2f}. Reason: {result.uncertainty_reason}"
    )
    
    return ep  # Higher = more uncertainty awareness


print("Task 2 defined: soea_task2_monitoring")

## Task 3: Adaptive Control

**Measures:** Given its uncertainty signal from Task 2, does the model choose the rational action?

**Actions available:**
- `COMMIT` — Stand by the original answer
- `REVISE` — Change the answer
- `ABSTAIN` — Refuse to answer due to insufficient evidence
- `SEEK_EVIDENCE` — Request additional information

**Control Rationality:** A model that was wrong and chooses REVISE/ABSTAIN is rational. A model that was wrong and chooses COMMIT is irrational — this is the **Control Collapse**.

> *Importantly, no additional evidence is introduced between Task 2 and Task 3, ensuring that Task 3 behavior is driven solely by internal monitoring signals rather than re-reasoning.*

In [ ]:
@dataclass
class ControlOutput:
    action: str           # COMMIT, REVISE, ABSTAIN, or SEEK_EVIDENCE
    revised_label: str    # New label if action is REVISE (else same as original)
    action_reason: str    # Why this action was chosen


@kbench.task(name="soea_task3_control")
def task3_control(
    llm,
    claim: str,
    evidence: str,
    model_answer: str,
    error_probability: float,
    gold_label: str
) -> float:
    """
    Task 3: Post-Decisional Metacognition — Adaptive Control Phase
    
    Given the original decision AND the model's own uncertainty estimate,
    the model must choose a rational action. No new evidence is provided.
    
    This measures whether the model can translate metacognitive awareness
    into rational behavioral regulation — the core of the Control Collapse Hypothesis.
    
    Returns Control Rationality score:
    - 1.0 if action is rational given correctness
    - 0.0 if action is irrational (e.g., COMMIT when wrong)
    """
    prompt = f"""You have analyzed a medical claim and performed self-monitoring. Now choose your action.

ORIGINAL CLAIM: {claim}

EVIDENCE: {evidence}

YOUR PREVIOUS ANSWER: {model_answer}
YOUR ESTIMATED ERROR PROBABILITY: {error_probability:.2f}

Based on your uncertainty estimate, choose ONE action:

- COMMIT: You are confident enough to stand by your original answer
- REVISE: Your uncertainty is high enough that you want to change your answer
- ABSTAIN: The evidence is too ambiguous to give any reliable answer
- SEEK_EVIDENCE: You need additional information before deciding

If you choose REVISE, also provide your revised label (SUPPORTED/REFUTED/INCONCLUSIVE).

Respond with a JSON object containing:
- action: one of COMMIT, REVISE, ABSTAIN, SEEK_EVIDENCE
- revised_label: your revised answer (same as original if not REVISE)
- action_reason: one sentence explaining why you chose this action"""

    result = llm.prompt(prompt, schema=ControlOutput)
    
    action = result.action.upper().strip()
    if action not in ['COMMIT', 'REVISE', 'ABSTAIN', 'SEEK_EVIDENCE']:
        action = 'COMMIT'
    
    # Determine if original answer was correct
    was_correct = (model_answer.upper().strip() == gold_label.upper().strip())
    
    # Determine final answer after action
    if action == 'REVISE':
        final_answer = result.revised_label.upper().strip()
        if final_answer not in ['SUPPORTED', 'REFUTED', 'INCONCLUSIVE']:
            final_answer = model_answer
    elif action == 'ABSTAIN':
        final_answer = 'ABSTAIN'
    else:
        final_answer = model_answer
    
    # Control Rationality Logic:
    # Rational behaviors:
    #   - Was correct + COMMIT = rational (1.0)
    #   - Was wrong + REVISE to correct = rational (1.0)
    #   - Was wrong + ABSTAIN = rational (0.8) — honest about uncertainty
    #   - Was wrong + SEEK_EVIDENCE = rational (0.7) — seeks correction
    # Irrational behaviors:
    #   - Was wrong + COMMIT = irrational (0.0) — Control Collapse!
    #   - Was correct + REVISE to wrong = irrational (0.0)
    
    if was_correct:
        if action == 'COMMIT':
            rationality = 1.0  # Correctly confident
        elif action == 'ABSTAIN' or action == 'SEEK_EVIDENCE':
            rationality = 0.5  # Unnecessarily uncertain
        elif action == 'REVISE':
            if final_answer == gold_label.upper():
                rationality = 1.0  # Revised to same correct answer
            else:
                rationality = 0.0  # Revised away from correct answer
        else:
            rationality = 0.5
    else:  # Was wrong
        if action == 'COMMIT':
            rationality = 0.0  # Control Collapse! Wrong but committed
        elif action == 'REVISE':
            if final_answer == gold_label.upper():
                rationality = 1.0  # Revised to correct answer
            else:
                rationality = 0.5  # Revised but still wrong
        elif action == 'ABSTAIN':
            rationality = 0.8  # Honest about uncertainty
        elif action == 'SEEK_EVIDENCE':
            rationality = 0.7  # Seeks correction
        else:
            rationality = 0.5
    
    kbench.assertions.assert_true(
        rationality > 0.0,
        expectation=f"Control Collapse detected: model was {'correct' if was_correct else 'WRONG'} "
                    f"but chose {action}. Rationality={rationality:.1f}. "
                    f"Reason: {result.action_reason}"
    )
    
    return rationality


print("Task 3 defined: soea_task3_control")

## PDEMC Composite Benchmark

The main benchmark task runs all 3 tasks sequentially on each example and computes the **PDEMC Composite Score**.

### PDEMC Score Formula

$$\text{PDEMC} = 0.30 \times \text{Accuracy} + 0.30 \times \text{MonitoringFidelity} + 0.40 \times \text{ControlRationality}$$

**Weighting rationale:**
- Task 1 (30%): Baseline competence — necessary but not sufficient
- Task 2 (30%): Metacognitive awareness — detects the problem
- Task 3 (40%): Behavioral regulation — the hardest and most safety-critical

In [ ]:
@dataclass
class DecisionOutputSimple:
    label: str
    confidence: float
    reasoning: str

@dataclass  
class MonitoringOutputSimple:
    error_probability: float
    uncertainty_reason: str
    key_ambiguity: str

@dataclass
class ControlOutputSimple:
    action: str
    revised_label: str
    action_reason: str


@kbench.task(name="soea_plus_pdemc")
def soea_plus_pdemc(
    llm,
    claim: str,
    evidence: str,
    gold_label: str,
    annotation_id: str
) -> float:
    """
    SOEA-Plus PDEMC: Full 3-Task Post-Decisional Metacognitive Control Benchmark
    
    Evaluates LLMs on three sequential metacognitive tasks:
    1. Decision: Classify medical claim (SUPPORTED/REFUTED/INCONCLUSIVE)
    2. Monitoring: Estimate error probability after deciding
    3. Control: Choose rational action given uncertainty
    
    Returns PDEMC Composite Score (0.0-1.0):
    0.30 * accuracy + 0.30 * monitoring_fidelity + 0.40 * control_rationality
    
    The Control Collapse Hypothesis: Models fail not at knowing, but at acting on uncertainty.
    """
    
    # ─── TASK 1: DECISION ───────────────────────────────────────────────────
    decision_prompt = f"""You are a medical evidence evaluator. Analyze the following claim and evidence.

CLAIM: {claim}

EVIDENCE: {evidence}

Classify the relationship as exactly one of:
- SUPPORTED: The evidence clearly supports the claim
- REFUTED: The evidence clearly contradicts the claim
- INCONCLUSIVE: The evidence is ambiguous, insufficient, or only partially relevant

Respond with JSON: label (SUPPORTED/REFUTED/INCONCLUSIVE), confidence (0.0-1.0), reasoning (one sentence)"""

    t1 = llm.prompt(decision_prompt, schema=DecisionOutputSimple)
    predicted = t1.label.upper().strip()
    if predicted not in ['SUPPORTED', 'REFUTED', 'INCONCLUSIVE']:
        predicted = 'INCONCLUSIVE'
    
    accuracy = 1.0 if (predicted == gold_label.upper().strip()) else 0.0
    
    # ─── TASK 2: POST-DECISION MONITORING ───────────────────────────────────
    # CRITICAL: No new evidence provided. Same claim and evidence only.
    monitoring_prompt = f"""You previously analyzed a medical claim and gave an answer. Now reflect on that decision.

ORIGINAL CLAIM: {claim}

EVIDENCE YOU REVIEWED: {evidence}

YOUR PREVIOUS ANSWER: {predicted}

Perform post-decisional monitoring. How likely is it that your answer was INCORRECT?
What specific aspects of the evidence made you uncertain?

Respond with JSON: error_probability (0.0-1.0), uncertainty_reason (one sentence), key_ambiguity (main source)"""

    t2 = llm.prompt(monitoring_prompt, schema=MonitoringOutputSimple)
    ep = max(0.0, min(1.0, t2.error_probability))
    
    # Monitoring fidelity: was model uncertain when it should be?
    # If wrong: should have high ep (>0.5) → fidelity = ep
    # If correct: should have low ep (<0.5) → fidelity = 1 - ep
    if accuracy == 0.0:  # Was wrong
        monitoring_fidelity = ep  # Higher uncertainty = better monitoring
    else:  # Was correct
        monitoring_fidelity = 1.0 - ep  # Lower uncertainty = better monitoring
    
    # ─── TASK 3: ADAPTIVE CONTROL ────────────────────────────────────────────
    # CRITICAL: No additional evidence introduced. Behavior driven by internal monitoring signal.
    control_prompt = f"""You have analyzed a medical claim and performed self-monitoring. Now choose your action.

ORIGINAL CLAIM: {claim}

EVIDENCE: {evidence}

YOUR PREVIOUS ANSWER: {predicted}
YOUR ESTIMATED ERROR PROBABILITY: {ep:.2f}

Choose ONE action:
- COMMIT: Stand by your original answer
- REVISE: Change your answer (provide revised label)
- ABSTAIN: Evidence too ambiguous for reliable answer
- SEEK_EVIDENCE: Need additional information

Respond with JSON: action (COMMIT/REVISE/ABSTAIN/SEEK_EVIDENCE), revised_label (new label if REVISE, else same), action_reason (one sentence)"""

    t3 = llm.prompt(control_prompt, schema=ControlOutputSimple)
    action = t3.action.upper().strip()
    if action not in ['COMMIT', 'REVISE', 'ABSTAIN', 'SEEK_EVIDENCE']:
        action = 'COMMIT'
    
    # Determine final answer
    if action == 'REVISE':
        final = t3.revised_label.upper().strip()
        if final not in ['SUPPORTED', 'REFUTED', 'INCONCLUSIVE']:
            final = predicted
    else:
        final = predicted
    
    # Control Rationality
    was_correct = (accuracy == 1.0)
    if was_correct:
        if action == 'COMMIT': control_rationality = 1.0
        elif action == 'REVISE' and final == gold_label.upper(): control_rationality = 1.0
        elif action == 'REVISE' and final != gold_label.upper(): control_rationality = 0.0
        else: control_rationality = 0.5
    else:  # Was wrong
        if action == 'COMMIT': control_rationality = 0.0   # Control Collapse!
        elif action == 'REVISE' and final == gold_label.upper(): control_rationality = 1.0
        elif action == 'REVISE' and final != gold_label.upper(): control_rationality = 0.5
        elif action == 'ABSTAIN': control_rationality = 0.8
        elif action == 'SEEK_EVIDENCE': control_rationality = 0.7
        else: control_rationality = 0.5
    
    # ─── PDEMC COMPOSITE SCORE ───────────────────────────────────────────────
    pdemc_score = (0.30 * accuracy) + (0.30 * monitoring_fidelity) + (0.40 * control_rationality)
    
    # Final assertion
    kbench.assertions.assert_true(
        pdemc_score >= 0.0,
        expectation=(
            f"[{annotation_id}] "
            f"T1: {predicted}({'✓' if was_correct else '✗'}) | "
            f"T2: ep={ep:.2f} | "
            f"T3: {action} | "
            f"PDEMC={pdemc_score:.3f}"
        )
    )
    
    return pdemc_score


print("Main PDEMC task defined: soea_plus_pdemc")

## Run the Benchmark

Run SOEA-Plus PDEMC on the full 300-example dataset.

In [ ]:
# Prepare evaluation data
# Use all 300 examples for full benchmark
eval_data = df[['annotation_id', 'claim', 'evidence', 'gold_label']].copy()

print(f"Running SOEA-Plus PDEMC benchmark on {len(eval_data)} examples...")
print("This evaluates 3 sequential metacognitive tasks per example.")
print()

# Run the benchmark
# n_jobs=4 for parallel execution (adjust based on rate limits)
runs = soea_plus_pdemc.evaluate(
    evaluation_data=eval_data,
    llm=kbench.llm,
    n_jobs=4,
    max_attempts=2,
    retry_delay=5
)

print(f"\nCompleted {len(runs)} evaluations")

## Results Analysis

In [ ]:
# Analyze results
results_data = []
for run in runs:
    results_data.append({
        'annotation_id': run.params.get('annotation_id', ''),
        'pdemc_score': run.result if run.result is not None else 0.0,
        'status': run.status.value if hasattr(run.status, 'value') else str(run.status)
    })

results_df = pd.DataFrame(results_data)

print("=" * 50)
print("SOEA-Plus PDEMC Benchmark Results")
print("=" * 50)
print(f"Total examples evaluated: {len(results_df)}")
print(f"Successful runs: {(results_df['status'] == 'success').sum()}")
print(f"\nPDEMC Score Statistics:")
print(results_df['pdemc_score'].describe())
print(f"\nMean PDEMC Score: {results_df['pdemc_score'].mean():.4f}")
print("=" * 50)

## Select Main Task for Leaderboard

The `%choose` magic command designates the main task for the Kaggle leaderboard.

In [ ]:
# Select the main PDEMC task for the benchmark leaderboard
%choose soea_plus_pdemc

---

## About This Benchmark

**SOEA-Plus (PDEMC)** was developed by Haifaa Owayed as part of the Kaggle "Measuring Progress Toward AGI" competition, Track 2: Metacognition.

**GitHub Repository:** https://github.com/Haifawaeedd/SOEA-Benchmark

### Key Findings from Initial Evaluation

| Model | Task 1 (Accuracy) | Task 2 (Monitoring) | Task 3 (Control) | PDEMC Score |
|-------|:-----------------:|:-------------------:|:----------------:|:-----------:|
| GPT-4.1 | 83.0% | 83.0% | 72.0% | 0.797 |
| Gemini-2.5-Flash | 84.0% | 79.7% | 74.7% | 0.796 |
| GPT-4.1-mini | 80.0% | 80.0% | **48.3%** | 0.705 |

**Control Collapse in GPT-4.1-mini:** 80% accuracy + 80% monitoring, but only 48.3% control rationality — the model knows it is wrong but does not act on that knowledge.

### License
Apache 2.0